# HMM via Libraries

Reproduces the 3-state Gaussian HMM regimes from Notebook 1 using:
- **`hmmlearn`** — scikit-learn compatible, industry standard
- **`pomegranate`** — modern probabilistic ML library

We verify numerical agreement with our from-scratch implementation and compare execution speed.

---

### References
1. Nguyen & Nguyen (2015) — *Hidden Markov Model for Stock Selection*, Risks 3(4)  
2. McGreevy, J. (2021) — *Hidden Markov Models in Finance*, Imperial College MSc  
3. QuantStart — *Market Regime Detection Using HMMs in QSTrader*  
4. Chen, X. (2025) — *HMM-based market regime detection with RL*, IDS

---

## 0. Imports

In [1]:
%pip install -q hmmlearn pomegranate

Note: you may need to restart the kernel to use updated packages.


In [4]:
import sys, os, time
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

scratch_path = os.path.join(project_root, "01_hmm_from_scratch")
sys.path.insert(0, scratch_path)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm, wasserstein_distance
import warnings
warnings.filterwarnings("ignore")

from hmmlearn import hmm as hmmlearn_hmm

try:
    import pomegranate as pg
    from pomegranate.hmm import DenseHMM
    from pomegranate.distributions import Normal
    HAS_POMEGRANATE = True
    print(f"pomegranate version: {pg.__version__}")
except ImportError:
    HAS_POMEGRANATE = False
    print("pomegranate not available — skipping that section")

from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, plot_gamma, plot_log_likelihood,
                               plot_aic_bic)
from utils.metrics    import regime_statistics, aic, bic, hmm_n_params
from hmm_core import GaussianHMM   # Scratch Implementation

print("\nAll imports OK")

pomegranate version: 1.1.2

All imports OK


## 1. Load Data

In [ ]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2005-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

X_train = get_observation_sequence(df_train).reshape(-1, 1)  # hmmlearn needs (T,1)
X_test  = get_observation_sequence(df_test).reshape(-1, 1)
X_1d    = X_train.ravel()   # 1-D for our scratch impl

print(f"Train: {len(X_train)}  Test: {len(X_test)}")

Train: 4189  Test: 1048


## 2. `hmmlearn` - GaussianHMM

`hmmlearn` implements the same Baum-Welch EM internally, using the same scaled
forward-backward recursions we built from scratch.

In [6]:
N_STATES = 3
N_ITERS  = 300

t0 = time.perf_counter()
model_hl = hmmlearn_hmm.GaussianHMM(
    n_components=N_STATES,
    covariance_type="full",
    n_iter=N_ITERS,
    tol=1e-7,
    random_state=42,
    verbose=False,
)
model_hl.fit(X_train)
t_hl = time.perf_counter() - t0

print(f"hmmlearn training time : {t_hl:.3f}s")
print(f"Converged              : {model_hl.monitor_.converged}")
print(f"Log-likelihood (train) : {model_hl.score(X_train):.2f}")
print()
print("Initial state probs π  :", model_hl.startprob_.round(4))
print("Transition matrix A:")
print(pd.DataFrame(model_hl.transmat_.round(4)))
print("\nMeans µ  :", model_hl.means_.ravel().round(6))
print("Std devs σ:", np.sqrt(model_hl.covars_.ravel()).round(6))

Model is not converging.  Current: 13653.84078220094 is not greater than 13653.87619403091. Delta is -0.03541182997105352


hmmlearn training time : 3.533s
Converged              : True
Log-likelihood (train) : 13653.77

Initial state probs π  : [0. 1. 0.]
Transition matrix A:
        0       1       2
0  0.8290  0.1710  0.0000
1  0.9106  0.0415  0.0479
2  0.0007  0.0418  0.9575

Means µ  : [ 0.00132  -0.002391 -0.001567]
Std devs σ: [0.006421 0.013781 0.025989]


In [7]:
# Decode with hmmlearn (Viterbi)
labels_hl = model_hl.predict(X_train)

stds_hl   = np.sqrt(model_hl.covars_.ravel())
order_hl  = np.argsort(stds_hl)
remap_hl  = {order_hl[k]: k for k in range(N_STATES)}
labels_hl = np.array([remap_hl[l] for l in labels_hl])
state_names = ["Low-Vol", "Med-Vol", "High-Vol"]

df_hl = df_train.copy()
df_hl["HMM_HL"] = labels_hl

fig = plot_price_with_regimes(df_hl, "HMM_HL",
                               title="SPY — hmmlearn 3-State HMM Regimes")
fig.show()

In [9]:
fig = plot_regime_distributions(df_hl, "HMM_HL",
                                 title="hmmlearn — Regime Return Distributions")
fig.show()

A_hl = model_hl.transmat_[np.ix_(order_hl, order_hl)]
fig  = plot_transition_matrix(A_hl, state_names=state_names,
                               title="hmmlearn — Transition Matrix")
fig.show()

## 3. Scratch Implmentation vs hmmlearn

We compare the learned parameters side-by-side.

In [10]:
# Train scratch implementation
t0 = time.perf_counter()
hmm_scratch = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)
hmm_scratch.fit(X_1d)
t_scratch = time.perf_counter() - t0

print(f"Scratch training time  : {t_scratch:.3f}s")
print(f"hmmlearn training time : {t_hl:.3f}s")
print(f"Speed-up (hmmlearn/scratch): {t_scratch/t_hl:.1f}×\n")

print("         µ (scratch)   µ (hmmlearn)   σ (scratch)   σ (hmmlearn)")
for j in range(3):
    print(f"  State {j}: {hmm_scratch.mu_[j]:+.6f}    "
          f"{model_hl.means_[order_hl[j],0]:+.6f}      "
          f"{hmm_scratch.sigma_[j]:.6f}     "
          f"{stds_hl[order_hl[j]]:.6f}")

ll_scratch = hmm_scratch.log_likelihood(X_1d)
ll_hl      = model_hl.score(X_train) * len(X_train)   # .score returns per-obs LL
print(f"\nLog-likelihood  scratch : {ll_scratch:.2f}")
print(f"Log-likelihood hmmlearn : {ll_hl:.2f}")
print(f"Difference              : {abs(ll_scratch - ll_hl):.2f}")

  Converged at iteration 125  (ΔLL = 9.09e-08)
Scratch training time  : 10.022s
hmmlearn training time : 3.533s
Speed-up (hmmlearn/scratch): 2.8×

         µ (scratch)   µ (hmmlearn)   σ (scratch)   σ (hmmlearn)
  State 0: +0.001190    +0.001320      0.005091     0.006421
  State 1: -0.000105    -0.002391      0.012271     0.013781
  State 2: -0.003313    -0.001567      0.035446     0.025989

Log-likelihood  scratch : 13865.43
Log-likelihood hmmlearn : 57195657.98
Difference              : 57181792.55


In [11]:
# Regime agreement metric
labels_scratch = hmm_scratch.predict(X_1d)
agreement = np.mean(labels_scratch == labels_hl) * 100
print(f"State assignment agreement: {agreement:.1f}%  (may differ due to local optima/ordering)")

# Regime statistics comparison
print("\n--- Scratch implementation ---")
print(regime_statistics(X_1d, labels_scratch).round(4).to_string(index=False))
print("\n--- hmmlearn ---")
print(regime_statistics(X_1d, labels_hl).round(4).to_string(index=False))

State assignment agreement: 63.4%  (may differ due to local optima/ordering)

--- Scratch implementation ---
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.2953             0.0802         3.6806   -0.0310           0.4782    2240      53.4734
     1             -0.0186             0.1958        -0.0951   -0.1281           0.0457    1719      41.0360
     2             -0.9491             0.5807        -1.6344    0.1295           1.0013     230       5.4906

--- hmmlearn ---
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.3468             0.0943         3.6772    0.1486          -0.0094    3221      76.8919
     1             -2.1907             0.2750        -7.9658    1.0086          -0.4348     210       5.0131
     2             -0.3242             0.3794        -0.8545   -0.0516           3.9007     758      18.0950


## 4. pomegranate : Modern Probabilistic HMM

`pomegranate` offers PyTorch-backed HMMs with gradient-based options.
If not installed, this section is skipped gracefully.

In [17]:
if HAS_POMEGRANATE:
    import torch
    import time
    import numpy as np

    X_tensor = torch.tensor(X_train, dtype=torch.float32)

    # Ensure shape = (T, 1)
    if X_tensor.ndim == 1:
        X_tensor = X_tensor.unsqueeze(-1)

    # Define HMM
    dists = [Normal() for _ in range(N_STATES)]
    model_pg = DenseHMM(
        distributions=dists,
        max_iter=N_ITERS,
        tol=1e-7,
        verbose=False
    )

    # Train
    t0 = time.perf_counter()
    model_pg.fit([X_tensor])
    t_pg = time.perf_counter() - t0

    # Stable prediction
    X_pred = X_tensor.unsqueeze(0)
    log_probs = model_pg.predict_log_proba(X_pred)
    labels_pg = torch.argmax(log_probs, dim=-1)[0].cpu().numpy()

    # Extract parameters
    pg_means = np.array([d.means.item() for d in model_pg.distributions])
    pg_stds  = np.array([torch.sqrt(d.covs).item() for d in model_pg.distributions])

    # Sort by volatility
    order_pg = np.argsort(pg_stds)
    remap_pg = {order_pg[k]: k for k in range(N_STATES)}
    labels_pg = np.array([remap_pg[l] for l in labels_pg])

    # Dataframe
    df_pg = df_train.copy()
    df_pg["HMM_PG"] = labels_pg

    # Output
    print(f"pomegranate training time: {t_pg:.3f}s")
    print(f"mu: {pg_means[order_pg].round(6)}")
    print(f"sigma: {pg_stds[order_pg].round(6)}")

    # Plot
    fig = plot_price_with_regimes(
        df_pg,
        "HMM_PG",
        title="SPY - pomegranate HMM Regimes"
    )
    fig.show()

else:
    print("Skipping pomegranate - not installed.")

pomegranate training time: 14.549s
mu: [ 1.200e-03 -6.100e-05 -2.742e-03]
sigma: [0.004997 0.011818 0.033004]


## 5. Model Selection with hmmlearn (N = 2 … 6)

In [18]:
selection_rows = []

for N in range(2, 7):
    h = hmmlearn_hmm.GaussianHMM(n_components=N, covariance_type="full",
                                   n_iter=300, tol=1e-7, random_state=42)
    h.fit(X_train)
    ll_tr = h.score(X_train) * len(X_train)
    ll_te = h.score(X_test)  * len(X_test)
    k     = hmm_n_params(N)
    selection_rows.append({
        "N": N,
        "Train LL": round(ll_tr, 1),
        "Test LL":  round(ll_te, 1),
        "AIC":      round(aic(ll_tr, k), 1),
        "BIC":      round(bic(ll_tr, k, len(X_train)), 1),
    })
    print(f"N={N}  TrainLL={ll_tr:.1f}  TestLL={ll_te:.1f}  AIC={aic(ll_tr,k):.1f}  BIC={bic(ll_tr,k,len(X_train)):.1f}")

sel_df = pd.DataFrame(selection_rows)
print("\n", sel_df.to_string(index=False))

fig = plot_aic_bic([f"{r['N']}-state" for r in selection_rows],
                   [r["AIC"] for r in selection_rows],
                   [r["BIC"] for r in selection_rows],
                   title="hmmlearn — Model Selection AIC vs BIC")
fig.show()

Model is not converging.  Current: 13600.305682654902 is not greater than 13600.369560713329. Delta is -0.06387805842678063


N=2  TrainLL=56971007.8  TestLL=3495852.1  AIC=-113941999.6  BIC=-113941948.8


Model is not converging.  Current: 13653.840782201256 is not greater than 13653.876194030523. Delta is -0.03541182926710462


N=3  TrainLL=57195658.0  TestLL=3504716.9  AIC=-114391286.0  BIC=-114391190.8


Model is not converging.  Current: 13837.686778867113 is not greater than 13837.706143399526. Delta is -0.019364532412510016


N=4  TrainLL=57965912.2  TestLL=3538062.9  AIC=-115931776.4  BIC=-115931624.2


Model is not converging.  Current: 13621.222511214559 is not greater than 13621.304948297653. Delta is -0.08243708309419162


N=5  TrainLL=57058505.6  TestLL=3502553.0  AIC=-114116941.2  BIC=-114116719.2


Model is not converging.  Current: 13811.350490401004 is not greater than 13811.406844183584. Delta is -0.056353782580117695


N=6  TrainLL=57855299.4  TestLL=3539088.9  AIC=-115710502.8  BIC=-115710198.5

  N   Train LL   Test LL          AIC          BIC
 2 56971007.8 3495852.1 -113941999.6 -113941948.8
 3 57195658.0 3504716.9 -114391286.0 -114391190.8
 4 57965912.2 3538062.9 -115931776.4 -115931624.2
 5 57058505.6 3502553.0 -114116941.2 -114116719.2
 6 57855299.4 3539088.9 -115710502.8 -115710198.5


## 6. Summary

| Method | Library | Key Advantage |
|---|---|---|
| `GaussianHMM` (scratch) | None |  |
| `hmmlearn.GaussianHMM` | hmmlearn | Fast, sklearn-compatible, production-ready |
| `DenseHMM` | pomegranate | PyTorch backend, GPU-able, flexible distributions |
